In [1]:
import numpy as np
import struct
from typing import Literal

HEADER_SIZE = 16 + 4 + 32 * 6   # 212
RECORD_SIZE = 88

According to the `Readme.rtf` file from the WeatherLink installation directory:

* The first 16 bytes are used to identify a weather database file and to identify different file formats.

* The next 4 bytes represent the Total Records, stored in a `long` variable type.

* The next `32*6` bytes represent the `dayIndex[32]` array. Each array has 2 bytes for `recordsInDay` and 4 bytes for `startPos`.
    * Empty arrays are represented as `(0,0)`
        * The first element of the array is always empty. According to the Readme, index 0 is not used.
        * For February, the last 3 elements will be empty if it is a non-leap year. For leap years, the last 2 elements will show as empty.
        * For months with 30 days, the last element of the array is always empty.
        * For months with 31 days, only the first element is empty.
    * With an archive interval of 15 minutes, the first record is the one at 00:15 on the first day of the month specified in the filename (`YYYY-MM.wlk`), and the final record is the one at 00:00 on the first day of the immediate next month.
        * **Example:** A weather station with an archive interval of 15 minutes has the file `2020-01.wlk`.
            * The first archive record has a time of `00:15` and represents the results recorded from `2020-01-01 00:00:00` to `2020-01-01 00:14:59`.
            * The last archive record has a time of `00:00` and represents the results recorded from `2020-01-31 23:45:00` to `2020-01-31 23:59:59`. On Weatherlink, this will show up as the first record/row of February 2020 (the specific date being `2020-02-01 00:00`).

After the header are a series of 88 bytes data records.

# Functions
The following block creates a bunch of functions to help in the transformation of data stored on the records.

In [2]:
def temperature_conversion(temp, unit : Literal["c","f"] = "c"):
    """
    Converts the temperature extracted from the .wlk file to the desired unit.

    The temperatures stored in the WLK file are stored as tenths of a degree F, therefore they must be divided by 10.

    Specify the unit to convert to with the `unit` argument (`f` for Fahrenheit, `c` for Celsius).
    By default, it will convert the temperature to celsius.
    """
    if temp == -32768:
        return None
    unit_clean = unit.lower()
    if unit_clean not in ("c","f"):
        raise ValueError("Unit must be exactly 'c' or 'f'.")
    return round(temp/10,1) if unit == "f" else round(((temp/10)-32)*(5/9),1)

def humidity_conversion(hum):
    """
    Converts the relative humidity extracted from the .wlk file.

    The relative humidity stored in the WLK file is stored as tenths of a percent, therefore it must be divided by 10.
    """
    return int(hum/10) if hum >= 0 else None

def dew_point_calculation(hum, out_temp, out_temp_unit: Literal["c","f"] = "c"):
    """
    Calculates the dew point with the provided relative humidity and current outside temperature.

    This function works with **Celsius temperatures** for both input and output.
     
    You may specify that the input temperature is on Fahrenheit with the `out_temp_unit` argument.
    By default, the function will assume the outside temperature provided is on Celsius units.
    """
    if out_temp == None or hum == None:
        return None

    unit_clean = out_temp_unit.lower()
    if unit_clean not in ("c","f"):
        raise ValueError("Unit must be exactly 'c' or 'f'.")
    
    if unit_clean == "f":
        out_temp = (out_temp-32)*(5/9)

    step_1 = np.log((hum/100) * np.exp((18.678-(out_temp/234.5))*(out_temp/(257.14+out_temp))))
    step_2 = (257.14*step_1)/(18.678-step_1)
    return round(float(step_2),2)

def wind_assign(wind_dir):
    """
    Assigns the direction bin to the extracted wind direction value of the record.

    If the wind direction value extracted is not part of the dictionary specified in this function (0-15), the Calm value will be assigned.
    """
    dirs = {
        0: "N",
        1: "NNE",
        2: "NE",
        3: "ENE",
        4: "E",
        5: "ESE",
        6: "SE",
        7: "SSE",
        8: "S",
        9: "SSW",
        10: "SW",
        11: "WSW",
        12: "W",
        13: "WNW",
        14: "NW",
        15: "NNW"
    }
    return dirs.get(wind_dir, "Calm")

def wind_speed_conversion(wind_speed, unit: Literal["mi","km"] = "km"):
    """
    Converts the wind speed extracted from the .wlk file to the desired unit.

    The wind speed stored in the WLK file is stored as tenths of a MPH, therefore it must be divided by 10.
    """
    unit_clean = unit.lower()
    if unit_clean not in ("mi", "km"):
        raise ValueError("Unit must be exactly 'mi' or 'km'.")
    
    return wind_speed/10 if unit == "mi" else round((wind_speed/10)*1.609,3)

def pressure_conversion(bar, unit: Literal["mb","hpa","mm","in"] = "mb"):
    """
    Converts the barometer reading extracted from the .wlk file to the desired unit.

    The pressure is stored as thousandths of an inch Hg, therefore it must be divided by 10.

    Specify the unit to convert to with the `unit` argument. Currently the units are:
        * Millibar (`mb`)
        * Hectopascal (`hpa`)
        * Millimeters of mercury (`mm`)
        * Inches of mercury (`in`)
    By default, it will convert into millibar.
    """
    unit_clean = unit.lower()
    if unit_clean not in ("mb","hpa","mm","in"):
        raise ValueError("Unit must be exactly 'mb', 'hpa', 'mm' or 'in'.")
    
    if unit_clean in ("mb", "hpa"):
        return (bar/1000)*33.864
    
    return bar/1000 if unit == "in" else (bar/1000)*25.4

def iss_reception_calculator(wind_samples: int, arc_int: int, wind_tx: int = 1):
    """
    Calculates the ISS Reception. This is the number of wind samples successfully received compared to an expected number.

    There's 8 transmitters available, and 7 different archive intervals. The calculated reception depends on the transmitter id and the archive interval of the weather station.
    """

    transmitters = {
        1: {1: 23, 5: 114, 10: 228, 15: 342, 30: 684, 60: 1368, 120: 2736},
        2: {1: 22, 5: 111, 10: 222, 15: 333, 30: 667, 60: 1335, 120: 2670},
        3: {1: 22, 5: 108, 10: 218, 15: 326, 30: 652, 60: 1302, 120: 2606},
        4: {1: 21, 5: 106, 10: 212, 15: 318, 30: 637, 60: 1273, 120: 2545},
        5: {1: 21, 5: 104, 10: 207, 15: 311, 30: 622, 60: 1244, 120: 2487},
        6: {1: 20, 5: 102, 10: 202, 15: 304, 30: 608, 60: 1216, 120: 2432},
        7: {1: 20, 5: 99, 10: 199, 15: 297, 30: 595, 60: 1189, 120: 2379},
        8: {1: 19, 5: 97, 10: 194, 15: 291, 30: 582, 60: 1165, 120: 2328}
    }

    if wind_tx not in transmitters:
        return -1
    
    return 1 if wind_samples / transmitters[wind_tx][arc_int] > 1 else wind_samples / transmitters[wind_tx][arc_int]

def rain_conversion(rain):
    """
    Converts the rain extracted from the .wlk file to proper millimeter (mm) reading.

    When no rain has fallen, it takes the value of either 8192 (0.2 mm units) or 4096 (0.01 inches) depending of the unit of measurement set in the program.
    Otherwise, rain is expressed as `code+x` where x is an integer value that stores the amount of rain fallen depending on the previous stated codes.
    """
    if rain in (4096,8192):
        return 0

    if rain > 8192:
        return (rain-8192)/5

    if rain > 4096:
        return ((rain-4096)*25.4)/100


def rain_rate_conversion(rain, rain_rate):
    """
    Converts the rain rate extracted from the .wlk file to proper millimeter per hour (mm/hr) reading.

    Currently this function only covers 0.2 mm and 0.01 inches rain collector types.
    """
    if rain_rate == 0:
        return 0
    if rain > 8192:
        return rain_rate/5
    if rain > 4096:
        return rain_rate/4


# Opening the .wlk file
The file used for this demostration is `2012-07.wlk`, which represents the data of July 2012 archived by Weatherlink.

In [3]:
with open("2012-07.wlk", "rb") as f:
    data = f.read()

id_code = data[:16]

total_records = struct.unpack_from("<I", data, 16)[0]

offset = 20

day_index = []

for day in range(32):
    records_in_day, start_pos = struct.unpack_from("<hI", data, offset)
    day_index.append((records_in_day, start_pos))
    offset += 6

print(f"ID Code (Bytes 0-15): {id_code}")
print(f"Total records (Bytes 16-19): {total_records}")
print(f"dayIndex[32] (Bytes 20-211): {day_index}")

ID Code (Bytes 0-15): b'WDAT5.3\x00\x00\x00\x00\x00\x00\x00\x05\x03'
Total records (Bytes 16-19): 3038
dayIndex[32] (Bytes 20-211): [(0, 0), (98, 0), (98, 98), (98, 196), (98, 294), (98, 392), (98, 490), (98, 588), (98, 686), (98, 784), (98, 882), (98, 980), (98, 1078), (98, 1176), (98, 1274), (98, 1372), (98, 1470), (98, 1568), (98, 1666), (98, 1764), (98, 1862), (98, 1960), (98, 2058), (98, 2156), (98, 2254), (98, 2352), (98, 2450), (98, 2548), (98, 2646), (98, 2744), (98, 2842), (98, 2940)]


# Initialize the dictionary
The following stats are stored in each record:
* Time
* Temperature Outside
* Highest Temperature
* Lowest Temperature
* Outside Humidity
* Wind Speed
* Wind Direction
* Highest Speed
* Highest Wind Direction
* Bar (pressure)
* Rain
* Rain Rate
* Solar Radiation
* Highest Solar Radiation
* Wind Samples
* Wind Transmitter ID (Wind Tx)
* Archive Interval

The following stats are not stored in the record, and therefore must be calculated with the former available retrieved data:
* Dew Point
* Wind Run
* Solar Energy
* ISS Reception

In [4]:
weather_record_template = {
    "day": 0,
    "time": "00:00",
    "temp_out": 255,
    "hi_temp": 255,
    "low_temp": 255,
    "out_hum": 255,
    "dew_pt": 255,
    "wind_speed": 255,
    "wind_dir": "DIR",
    "wind_run": 255,
    "hi_speed": 255,
    "hi_dir": "DIR",
    "bar": 9999,
    "rain": -1,
    "rain_rate": -1,
    "solar_rad": 9999,
    "solar_energy": 9999,
    "hi_solar_rad": 9999,
    "wind_samp": 9999,
    "wind_tx": -1,
    "iss_reception": -1,
    "arc_int": -1,
}

# Storing the data in the dictionary
According to the `Readme.rtf` file, these are (some of) the following stats stored in the WLK file:

<center>

|       **Stat**       |                     **Description**                    |
|:--------------------:|:------------------------------------------------------:|
|       dataType       |               1 (standard archive record)              |
|    archiveInterval   |            Number of minutes in the archive            |
|       iconFlags      |    Icon associated with this record, plus Edit flags   |
|       moreFlags      |                       Tx Id, etc.                      |
|      packedTime      | Minutes past midnight of the end of the archive period |
|      outsideTemp     |                  tenths of a degree F                  |
|     hiOutsideTemp    |                  tenths of a degree F                  |
|    lowOutsideTemp    |                  tenths of a degree F                  |
|      insideTemp      |                  tenths of a degree F                  |
|       barometer      |                thousandths of an inch Hg               |
|      outsideHum      |                   tenths of a percent                  |
|       insideHum      |                   tenths of a percent                  |
|         rain         |       number of clicks + rain collector type code      |
|      hiRainRate      |                     clicks per hour                    |
|       windSpeed      |                    tenths of an MPH                    |
|      hiWindSpeed     |                    tenths of an MPH                    |
|     windDirection    |               direction code (0-15, 255)               |
|    hiWindDirection   |               direction code (0-15, 255)               |
|    numWindSamples    |    number of valid ISS packets containing wind data    |
| solarRad, hiSolarRad |                 Watts per meter squared                |

</center>

Therefore, it's necessary to perform some transformations in the data that will be stored in the dictionary once extracted from the record.

This is possible thanks to the previously created functions.

In [5]:
'''
day_index[x][1] is the start_pos of the day x
There's 1 record that should not be read.
'''

for day in range(1,len(day_index)):
    records_in_day, start_pos = day_index[day]
    if records_in_day == 0:
        continue
    for j in range(records_in_day-2):
        record_index = start_pos + 2 + j
        record = data[
            HEADER_SIZE + record_index * RECORD_SIZE :
            HEADER_SIZE + (record_index + 1) * RECORD_SIZE
        ]
        (
        data_type,
        archive_interval,
        icon_flags,
        more_flags,

        packed_time,
        outside_temp,
        hi_outside_temp,
        low_outside_temp,
        inside_temp,
        barometer,
        outside_hum,
        inside_hum,

        rain_code,
        hi_rain_rate,

        wind_speed,
        hi_wind_speed,
        wind_dir,
        hi_wind_dir,

        num_wind_samples,

        solar_rad,
        hi_solar_rad,
        ) = struct.unpack_from(
            "<BBBBhhhhhhhhHhhhBBhhh",
            record,
            0
        )

        if data_type != 1:
            print("Invalid record type!")
            break

        weather_record = weather_record_template.copy()
        weather_record.update({
            "day": day+1 if (packed_time//60 == 24 and packed_time%60 == 0) else day,
            "time": f"{"0" if (packed_time//60 == 24) else packed_time//60}" + ":" + f"{"00" if packed_time%60 == 0 else packed_time%60}",
            "temp_out": temperature_conversion(outside_temp),
            "hi_temp": temperature_conversion(hi_outside_temp),
            "low_temp": temperature_conversion(low_outside_temp),
            "out_hum": humidity_conversion(outside_hum),
            "dew_pt": dew_point_calculation(humidity_conversion(outside_hum), temperature_conversion(outside_temp)),
            "wind_speed": wind_speed_conversion(wind_speed),
            "wind_dir": wind_assign(wind_dir),
            "wind_run": round(wind_speed_conversion(wind_speed) * archive_interval / 60,2),
            "hi_speed": wind_speed_conversion(hi_wind_speed),
            "hi_dir": wind_assign(hi_wind_dir),
            "bar": round(pressure_conversion(barometer, unit='hpa'),2),
            "rain": rain_conversion(rain_code),
            "rain_rate": rain_rate_conversion(rain_code, hi_rain_rate),
            "solar_rad": solar_rad,
            "solar_energy": round((solar_rad*archive_interval*60)/41840,2),
            "hi_solar_rad": None if hi_solar_rad == 32767 else hi_solar_rad,
            "wind_samp": num_wind_samples,
            "wind_tx": more_flags+1,
            "iss_reception": round(iss_reception_calculator(num_wind_samples, archive_interval, more_flags+1),3),
            "arc_int": archive_interval
        })
print(f"Last record: {weather_record}")


Last record: {'day': 32, 'time': '0:00', 'temp_out': 3.0, 'hi_temp': 3.2, 'low_temp': 2.8, 'out_hum': 90, 'dew_pt': 1.52, 'wind_speed': 0.0, 'wind_dir': 'E', 'wind_run': 0.0, 'hi_speed': 1.609, 'hi_dir': 'E', 'bar': 1017.99, 'rain': 0, 'rain_rate': 0, 'solar_rad': 0, 'solar_energy': 0.0, 'hi_solar_rad': 0, 'wind_samp': 352, 'wind_tx': 1, 'iss_reception': 1, 'arc_int': 15}


As a summary, the stats and their respective units stored in the previous cell are as follows:

<center>

|         **Stat**        |         **Unit**        |
|:-----------------------:|:-----------------------:|
|           Day           |           N/A           |
|           Time          |           N/A           |
|   Temperature Outside   |         Celsius         |
|   Highest Temperature   |         Celsius         |
|    Lowest Temperature   |         Celsius         |
|     Outside Humidity    |        Percentage       |
|        Dew Point        |         Celsius         |
|        Wind Speed       |   Kilometers per hour   |
|      Wind Direction     |           N/A           |
|         Wind Run        |        Kilometers       |
|      Highest Speed      |   Kilometers per hour   |
|    Highest Direction    |           N/A           |
|           Bar           | Millibar or Hectopascal |
|          Rain           |        Millimeters      |
|        Rain Rate        |   Millimeters per hour  |
|     Solar Radiation     | Watts per meter squared |
|       Solar Energy      |         Langley         |
| Highest Solar Radiation | Watts per meter squared |
|       Wind Samples      |           N/A           |
|         Wind Tx         |           N/A           |
|      ISS Reception      |           N/A           |
|     Archive Interval    |         Minutes         |

</center>


# Comments

As is, this procedure can be improved further upon.
* The final record will always have in the `day` field the value `day+1`, so you can expect:
    * February with last record reporting the `day` as `29` or `30`
    * Months with 31 days with a final record with `'day': 32`
* Rain stats extracted here currently only covers two sensors: 0.01 inch and 0.2 mm. The reason behind it it's plain simple: the data I'm working with uses those.
    * Other codes according to the documentation: `0` (0.1 inch), `12288` (1.0 mm) and `24576` (0.1 mm)
* The procedure assumes the record type is always 1 (standard archive record).
* UV stats are part of the record structure, but are excluded during the extraction since the current project scope does not need them.
* The indexes (Wind Chill, Heat, THW, THSW) are not included since **I do not know the *exact* formulas** used for those. Although we have the values, it's of no use if we do not know how to properly calculate the desired index.